RDDs — Resilient Distributed Datasets — are the original Spark data abstraction. Every DataFrame and Dataset you write today compiles down to RDD operations under the hood. Understanding RDDs explains *why* Spark behaves the way it does: lazy evaluation, fault tolerance through lineage, and partition-level parallelism.

## What is an RDD?

An RDD is defined by three properties that its name describes:

| Property | Meaning |
|---|---|
| **Resilient** | Fault-tolerant — lost partitions are recomputed from lineage, not from replicated data |
| **Distributed** | Partitioned across executor memory on multiple nodes |
| **Dataset** | A collection of records — can be any JVM object (strings, tuples, case classes) |

Additional characteristics:
- **Immutable** — you never modify an RDD; transformations produce a new RDD
- **Lazily evaluated** — transformations are recorded but not executed until an action is called
- **Typed** — in Python, `RDD[Row]`; in Scala, fully type-parameterised
- **Schema-free** — unlike DataFrames, RDDs have no built-in schema or column names

## Partitions — the Unit of Parallelism

An RDD is split into **partitions** — chunks of data that can be processed independently and in parallel. Each partition is processed by one task on one executor core.

![RDD Partitions](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/rdd-partitions.svg)

- The number of partitions determines the **degree of parallelism** for that RDD
- By default, `sc.parallelize(data)` creates as many partitions as `sc.defaultParallelism` (usually equal to total executor cores)
- You can control partitions explicitly: `sc.parallelize(data, numSlices=8)`
- After a shuffle, the number of output partitions is controlled by `spark.sql.shuffle.partitions` (default 200)

## Lineage & Fault Tolerance

Spark does **not** replicate RDD data to achieve fault tolerance. Instead, it records the sequence of transformations that produced each RDD — the **lineage graph**. If a partition is lost (executor crash), Spark replays only the relevant portion of the lineage to recompute it.

![RDD Lineage](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/rdd-lineage.svg)

This is why **caching matters**: if you reuse an RDD multiple times without caching it, Spark recomputes it from the source each time. Cache it with `.cache()` or `.persist()` to materialise it in memory once.

## Creating RDDs

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDDs")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
print(f"Default parallelism: {sc.defaultParallelism}")

In [ ]:
# 1. From a Python collection
rdd1 = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8], numSlices=4)
print(f"Partitions: {rdd1.getNumPartitions()}")
print(f"Data      : {rdd1.collect()}")

In [ ]:
# 2. From a text file — each line becomes one element
# rdd_file = sc.textFile("path/to/file.txt")

# Simulated equivalent
lines = ["hello world", "spark is fast", "hello spark", "rdd is the foundation"]
rdd_lines = sc.parallelize(lines)
print(f"Lines: {rdd_lines.collect()}")

In [ ]:
# 3. From a DataFrame (when you need low-level control)
df = spark.range(10)
rdd_from_df = df.rdd
print(type(rdd_from_df))
print(rdd_from_df.take(3))  # Row objects

## RDD Transformations

Like DataFrames, RDD transformations are **lazy**.

| Transformation | Description |
|---|---|
| `map(f)` | Apply `f` to each element, one-to-one |
| `flatMap(f)` | Apply `f` to each element, one-to-many (flattens result) |
| `filter(f)` | Keep elements where `f` returns `True` |
| `distinct()` | Remove duplicates (causes a shuffle) |
| `union(rdd2)` | Concatenate two RDDs |
| `reduceByKey(f)` | Merge values for each key using `f` (Pair RDD) |
| `groupByKey()` | Group all values for each key (shuffles all values — prefer `reduceByKey`) |
| `sortByKey()` | Sort by key (causes a shuffle) |
| `mapValues(f)` | Apply `f` only to values in a key-value RDD |
| `repartition(n)` | Reshuffle data into `n` partitions (full shuffle) |
| `coalesce(n)` | Reduce to `n` partitions without a full shuffle |

In [ ]:
# map — square every number
squared = rdd1.map(lambda x: x ** 2)
print("Squared:", squared.collect())

# filter — keep even numbers only
evens = rdd1.filter(lambda x: x % 2 == 0)
print("Evens  :", evens.collect())

# flatMap — split each sentence into words
words = rdd_lines.flatMap(lambda line: line.split(" "))
print("Words  :", words.collect())

## Key-Value (Pair) RDDs

A **Pair RDD** is an RDD of `(key, value)` tuples. It unlocks a set of aggregation and grouping operations that parallel what SQL GROUP BY does.

In [ ]:
# Word count — the classic Spark example
word_counts = (
    rdd_lines
    .flatMap(lambda line: line.split(" "))   # split into words
    .map(lambda word: (word, 1))             # emit (word, 1) pairs
    .reduceByKey(lambda a, b: a + b)         # sum counts per word
    .sortBy(lambda kv: kv[1], ascending=False)
)
word_counts.collect()

In [ ]:
# reduceByKey vs groupByKey
# reduceByKey: partial aggregation on each partition BEFORE shuffle — much less data transferred
# groupByKey:  shuffles ALL values first, then aggregates — avoid for large datasets

pairs = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("b", 4), ("c", 5)])

# Preferred
by_key = pairs.reduceByKey(lambda a, b: a + b)
print("reduceByKey:", sorted(by_key.collect()))

# Works but shuffles more data
grouped = pairs.groupByKey().mapValues(list)
print("groupByKey :", sorted(grouped.collect()))

## RDD Actions

Actions trigger execution and return a result to the driver or write to storage.

| Action | Returns |
|---|---|
| `collect()` | All elements as a Python list — careful with large RDDs |
| `count()` | Number of elements |
| `first()` | First element |
| `take(n)` | First `n` elements |
| `top(n)` | Top `n` elements (descending order) |
| `reduce(f)` | Aggregate all elements using `f` |
| `countByValue()` | Dict of `{value: count}` |
| `saveAsTextFile(path)` | Write one file per partition to a path |
| `foreach(f)` | Apply `f` to each element on executors (side effects only) |

In [ ]:
print("count     :", rdd1.count())
print("first     :", rdd1.first())
print("take(3)   :", rdd1.take(3))
print("top(3)    :", rdd1.top(3))
print("reduce sum:", rdd1.reduce(lambda a, b: a + b))
print("countByVal:", words.countByValue())

## Persistence — `cache()` and `persist()`

By default, Spark recomputes an RDD from scratch every time an action is called on it. If you reuse the same RDD more than once, **persist it**.

In [ ]:
from pyspark import StorageLevel

# .cache() == .persist(StorageLevel.MEMORY_ONLY)
word_counts.cache()

print(word_counts.count())   # materialises into memory
print(word_counts.take(3))   # served from cache — no recomputation

# Storage levels
# MEMORY_ONLY          — default; spills nothing (data lost if evicted)
# MEMORY_AND_DISK      — spills to disk on eviction; safe for large RDDs
# DISK_ONLY            — always on disk; slowest but lowest memory cost
# MEMORY_ONLY_2        — replicates to 2 nodes (fault tolerant, expensive)

word_counts.unpersist()  # release when done

## RDD vs DataFrame — When to Use Each

| | RDD | DataFrame |
|---|---|---|
| **API level** | Low-level, explicit | High-level, declarative |
| **Optimisation** | Manual — you control the plan | Catalyst auto-optimises |
| **Performance** | Slower (no Tungsten codegen) | Faster for structured data |
| **Type safety** | Python: untyped `object` | Column-typed with schema |
| **Schema** | None | Required |
| **Use when** | Unstructured data, custom serialisation, ML feature engineering on arbitrary objects, calling `mapPartitions` for custom I/O | Almost everything else |

> **Guideline**: prefer the DataFrame API. Drop to RDDs only when you need something the DataFrame API cannot express — for example, complex custom logic per-partition with `mapPartitions`, or working with non-tabular data.

In [ ]:
# mapPartitions — apply a function once per partition (not once per row)
# Useful for expensive setup (e.g. open a DB connection per partition, not per row)
def process_partition(records):
    # Imagine opening a DB connection here
    for record in records:
        yield record * 10  # process each record
    # Connection would close here

result = rdd1.mapPartitions(process_partition)
print("mapPartitions:", result.collect())

## Summary

- An RDD is **Resilient** (lineage-based fault tolerance), **Distributed** (partitioned across executors), and a **Dataset** (collection of any JVM object).
- RDDs are **immutable** and **lazily evaluated** — transformations build a lineage graph; actions trigger execution.
- **Partitions** are the unit of parallelism — one task per partition per executor core.
- Fault tolerance works by **replaying lineage**, not replicating data — cache RDDs you reuse to avoid recomputation.
- **Pair RDDs** (`(key, value)` tuples) enable `reduceByKey`, `groupByKey`, `mapValues`, and `sortByKey` — prefer `reduceByKey` over `groupByKey` to minimise shuffle data.
- Use `.cache()` or `.persist(StorageLevel.MEMORY_AND_DISK)` for RDDs used more than once.
- In modern Spark, prefer the **DataFrame API** — it is faster and auto-optimised. Use RDDs only for low-level control that DataFrames cannot provide.